# BERT Pre-Fine-Tuning on OWS Unlabeled Corpus (Masked Language Modeling)

This notebook performs domain-adaptive pre-training (masked language modeling, MLM) of BERT models
on large-scale unlabeled web texts from the OpenWebSearch (OWS) corpus.

**Pre-fine-tuning** means continuing BERT's masked language modeling objective on in-domain data
before task-specific fine-tuning on labeled hate speech datasets. This step is shown to improve
downstream classification performance (Gururangan et al., 2020).

**Four model variants are trained**, each on a different language subset of OWS:
- `OwsEng` — English texts
- `OwsDeu` — German texts
- `OwsSpa` — Spanish texts
- `Ows4L` — All four languages combined (2.7M texts)

**Input datasets** (loaded from Hugging Face):
Raw:
-    4L — 4-language OWS corpus (DEU, ENG, SPA, VIE)
-    deu — German unlabeled subset
-    eng — English unlabeled subset
-    spa — Spanish unlabeled subset

**Output**: Pre-fine-tuned BERT checkpoints saved locally and uploaded to Hugging Face.

In [ ]:
import pandas as pd
import json
import os
import numpy as np
from datasets import Dataset

from transformers import (
    set_seed,
    TrainingArguments, 
    Trainer,
    DataCollatorForLanguageModeling,
    AutoModelForMaskedLM,
    AutoTokenizer,
)

from time import time
import pickle
import matplotlib.pyplot as plt
import random
from tqdm import tqdm
import torch

def_seed = 42

set_seed(def_seed)
np.random.seed(def_seed)
import random
random.seed(def_seed)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

from sklearn.model_selection import train_test_split


## 2. Define MLM Pre-Fine-Tuning Function

**`pretrain_Bert`**: Implements masked language modeling (MLM) continuation pre-training for a BERT model.
The function:
1. Tokenizes and chunks the input dataframe texts into 512-token windows
2. Applies `DataCollatorForLanguageModeling` with 15% random token masking
3. Wraps Hugging Face `Trainer` with configurable training arguments
4. Saves the resulting checkpoint to `save_path`

This function is called once per language variant (OwsEng, OwsDeu, OwsSpa, Ows4L).

In [ ]:

def pretrain_Bert(
        df=None,
        base=None,
        tokenizer=None,
        save_path=None,
):
    
    df = df.sample(frac=1, random_state=def_seed).reset_index(drop=True)

    dataset = Dataset.from_pandas(df)
    
    def tokenize_and_chunk(example):
        outputs = tokenizer(
            example["text"],
            truncation=True,
            max_length=512,
            stride=0,  
        )

        result = {
            "input_ids": outputs["input_ids"],
            "attention_mask": outputs["attention_mask"],
        }

        return result


    tokenized_dataset = dataset.map(
        tokenize_and_chunk,
        batched=True,
        remove_columns=["text"],  
        num_proc=10,
    )
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=0.15,
    )

    training_args = TrainingArguments(
        output_dir=save_path,
        overwrite_output_dir=True,
        num_train_epochs=1,
        save_strategy="steps",        
        save_steps=3000,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=1,
        learning_rate=1e-5,
        lr_scheduler_type="cosine",
        warmup_ratio=0.2,
        max_grad_norm=0.2,
        logging_steps=200,
        fp16=False,
        bf16=True,
        gradient_accumulation_steps=2,
        gradient_checkpointing=True,
        report_to="none",              
        eval_steps=5000,
        max_steps=-1,                  
        log_level="debug",
        save_total_limit=10,
        dataloader_num_workers=16,
    )



    trainer = Trainer(
        model=base,
        args=training_args,
        train_dataset=tokenized_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,

    )
    # trainer.train()
    trainer.train(resume_from_checkpoint=True)
    return trainer

## 3. Load OWS Unlabeled Datasets from Hugging Face

Load the four OWS unlabeled text corpora from Hugging Face. Each dataset was collected from
OpenWebSearch and filtered via URL/schema.org metadata to target conversational web content:

Select the desired dataset in the next cell to build the training dataframe.

In [ ]:



from datasets import load_dataset

repo = "danghaidang-passau/HateOWS-dataset-LREC2026"
ds_raw = load_dataset(repo, "raw")       

# access splits:
ds_4l = ds_raw["4L"]
datataset_Deu = ds_raw["deu"]
datataset_Eng = ds_raw["eng"]
datataset_Spa = ds_raw["spa"]


## 4. Build Training DataFrame

Convert the loaded Hugging Face dataset split to a pandas DataFrame. Use the `['train']` split which
contains all texts for that language variant. Change the dataset variable to switch between language
configurations (e.g., `datataset_Eng`, `datataset_Deu`, `datataset_Spa`, or `datataset_4L_2_7M`).

In [ ]:
df = ds_4l.to_pandas()

## 5. Load BERT Base Model and Tokenizer

Load the base BERT model and its tokenizer from Hugging Face. The base is `google-bert/bert-base-uncased`
for English, German, and multilingual variants, or a language-specific BERT for other configurations.
The tokenizer is used for MLM tokenization inside `pretrain_Bert`.

In [ ]:
model_name = "google-bert/bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)

## 6. Run Pre-Fine-Tuning

Call `pretrain_Bert` to start MLM pre-fine-tuning on the selected OWS corpus. The trained checkpoint
will be saved to the `save_path` directory (e.g., `"Ows4L"`). Training time depends on corpus size;
the 4-language 2.7M-text corpus takes approximately 6-8 hours on a single A100 GPU.

After training, upload the checkpoint to Hugging Face (`danghaidang-passau/Ows4L_16`, etc.) for use
in `eval_LLMs.ipynb`.

In [ ]:
trainer = pretrain_Bert(
    df=df,
    base=model,
    tokenizer=tokenizer,
    save_path="Ows4L"
)